# QUERY 2
Nombre de banco, cuenta de origen y monto de la max. transacción USD de cada banco.

In [6]:
# importamos dataset
import pandas as pd

CANTIDAD_FILAS = 9
RUTA_DE_DATASETS = "../../datasets"
NOMBRE_ARCHIVO_TRANSACCIONES = "transacciones_sample_pequeño.csv"
NOMBRE_ARCHIVO_CUENTAS = "accounts_sample_pequeño.csv"
RUTA_TRANSACCIONES_SAMPLE = f"{RUTA_DE_DATASETS}/transacciones_sample.csv"
RUTA_CUENTAS_SAMPLE = f"{RUTA_DE_DATASETS}/accounts_sample.csv"

transacciones_completas = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO_TRANSACCIONES}",
    dtype={
        "From Bank": "string",
        "Account": "string",
        "To Bank": "string",
        "Account.1": "string"
    }
)
cuentas_completas = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO_CUENTAS}",
    dtype={
        "Bank ID": "string",
        "Account Number": "string"
    }
)
columnas_cuentas_originales = cuentas_completas.columns.tolist()

def normalizar_bank_id(serie):
    return serie.str.strip().str.lstrip("0").replace("", "0")

transacciones = transacciones_completas.sample(n=CANTIDAD_FILAS, random_state=42)
guardar_csv = lambda df, nombre_archivo: df.to_csv(nombre_archivo, index=False)
guardar_csv(transacciones, RUTA_TRANSACCIONES_SAMPLE)

transacciones_sample = pd.read_csv(
    RUTA_TRANSACCIONES_SAMPLE,
    dtype={
        "From Bank": "string",
        "Account": "string",
        "To Bank": "string",
        "Account.1": "string"
    }
)

transacciones_sample["From Bank Normalized"] = normalizar_bank_id(transacciones_sample["From Bank"])
transacciones_sample["To Bank Normalized"] = normalizar_bank_id(transacciones_sample["To Bank"])
cuentas_completas["Bank ID Normalized"] = normalizar_bank_id(cuentas_completas["Bank ID"])

cuentas_relevantes = pd.concat([
    transacciones_sample[["From Bank Normalized", "Account"]].rename(
        columns={"From Bank Normalized": "Bank ID Normalized", "Account": "Account Number"}
    ),
    transacciones_sample[["To Bank Normalized", "Account.1"]].rename(
        columns={"To Bank Normalized": "Bank ID Normalized", "Account.1": "Account Number"}
    )
], ignore_index=True).dropna().drop_duplicates()

cuentas_sample = cuentas_completas.merge(
    cuentas_relevantes,
    on=["Bank ID Normalized", "Account Number"],
    how="inner"
)[columnas_cuentas_originales]
guardar_csv(cuentas_sample, RUTA_CUENTAS_SAMPLE)

cuentas_sample.head()

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Banco Único S.A.,123651,80ED98180,800F49110,Sole Proprietorship #43351
1,Banco Único S.A.,123651,80A36EE80,800964DC0,Partnership #51172
2,Banco Único S.A.,123651,80CA95580,8005F8A20,Corporation #16457
3,Banco Único S.A.,123651,80B77AA10,8001ABCD0,Corporation #90001
4,Banco tania.,123652,80ED98180,800F49110,Sole Proprietorship #43351


In [7]:
RUTA_RESULTADO_QUERY2 = f"{RUTA_DE_DATASETS}/query2_result.csv"

transacciones_sample = pd.read_csv(
    RUTA_TRANSACCIONES_SAMPLE,
    dtype={
        "From Bank": "string",
        "Account": "string",
        "To Bank": "string",
        "Account.1": "string"
    }
)
cuentas_sample = pd.read_csv(
    RUTA_CUENTAS_SAMPLE,
    dtype={
        "Bank ID": "string",
        "Account Number": "string"
    }
)

transacciones_usd = transacciones_sample[
    transacciones_sample["Payment Currency"] == "US Dollar"
].copy()
transacciones_usd["From Bank Normalized"] = normalizar_bank_id(transacciones_usd["From Bank"])
transacciones_usd["Amount Paid"] = pd.to_numeric(transacciones_usd["Amount Paid"], errors="coerce")
transacciones_usd = transacciones_usd.dropna(subset=["Amount Paid"])

# Un resultado por Bank ID (único): máximo de Amount Paid y la cuenta que lo generó
bancos = cuentas_sample[["Bank ID", "Bank Name"]].copy()
bancos["From Bank Normalized"] = normalizar_bank_id(bancos["Bank ID"])
bancos = bancos[["From Bank Normalized", "Bank ID", "Bank Name"]].drop_duplicates(subset=["From Bank Normalized"])

transacciones_con_banco = transacciones_usd.merge(bancos, on="From Bank Normalized", how="inner")

idx_maximos = transacciones_con_banco.groupby("From Bank Normalized")["Amount Paid"].idxmax()
resultado_query2 = transacciones_con_banco.loc[idx_maximos, ["Bank ID", "Bank Name", "Account", "Amount Paid"]].rename(columns={
    "Account": "Cuenta Origen",
    "Amount Paid": "Monto Max USD"
}).sort_values(by="Bank ID").reset_index(drop=True)

guardar_csv(resultado_query2, RUTA_RESULTADO_QUERY2)
resultado_query2.head()

,Bank ID,Bank Name,Cuenta Origen,Monto Max USD
0,123651,Banco Único S.A.,80A36EE80,16946.26
1,123652,Banco tania.,80CA95580,51772.13
2,BANK_1,Banco 2,80B77AA10,73361.71
